# DaSiWa MiniMax H3 auf RunPod - Steuerung

Dieses Notebook ist optional. Damit kannst du
* den Status ansehen (Provisioning-, Download- und ComfyUI-Log),
* **LoRAs von CivitAI nachladen**, ohne den Pod neu zu starten,
* **ComfyUI neu starten**.

Führe zuerst die Zelle **0** aus.

## 0 · Vorbereitung

In [ ]:
import os, subprocess, getpass
WS = "/workspace" if os.access("/workspace", os.W_OK) else "/root"
BIN = f"{WS}/dasiwa/bin"

def run(cmd, env=None):
    """Fuehrt einen Shell-Befehl aus und zeigt die Ausgabe live an."""
    e = dict(os.environ, **(env or {}))
    p = subprocess.Popen(["bash", "-lc", cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=e)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

print("Workspace:", WS)
print("CIVITAI_TOKEN gesetzt:", bool(os.environ.get("CIVITAI_TOKEN") or os.environ.get("civitai_token")))

## 1 · Status

In [ ]:
run("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv")
print("\n--- Provisioning ---"); run(f"tail -n 15 {WS}/dasiwa/provision.log")
print("\n--- Downloads ---");    run(f"tail -n 15 {WS}/dasiwa/download.log")
print("\n--- ComfyUI ---");      run(f"tail -n 15 {WS}/comfyui.log")

## 2 · LoRAs von CivitAI laden

Trage die LoRAs in `LORAS` ein, durch Komma getrennt. Erlaubt sind:
* **Model-Version-ID**, also die Zahl hinter `modelVersionId=` in der CivitAI-URL, z. B. `3377709`
* eine CivitAI-URL: `https://civitai.com/models/2979777` (nimmt die neueste Version) oder `…?modelVersionId=…`
* eine Hugging-Face- oder andere direkte URL
* optional mit eigenem Dateinamen: `3377709|meine_lora.safetensors`

Ist `CIVITAI_TOKEN` im Template gesetzt, wird er automatisch benutzt. Sonst fragt die Zelle danach.

In [ ]:
LORAS = "3377709"      # <- hier eintragen, z. B. "3377709, https://civitai.com/models/2929860"
UNTERORDNER = ""        # optional, z. B. "MiniMaxH3" -> models/loras/MiniMaxH3/

token = os.environ.get("CIVITAI_TOKEN") or os.environ.get("civitai_token") or getpass.getpass("CivitAI API-Key (leer = ohne): ")
run(f"{BIN}/download_models.sh", env={"SKIP_BASE_MODELS": "1", "CIVITAI_LORAS": LORAS, "CIVITAI_TOKEN": token, "LORA_SUBDIR": UNTERORDNER})
print("\nFertig. Im ComfyUI-Browser R druecken (oder Zelle 4 ausfuehren), damit die LoRAs in der Liste erscheinen.")

## 3 · Vorhandene LoRAs anzeigen

In [ ]:
run(f"cd {WS}/ComfyUI/models/loras && find . -type f \\( -name '*.safetensors' -o -name '*.ckpt' -o -name '*.pt' \\) -printf '%10s  %p\\n' | sort -k2 | numfmt --field=1 --to=iec")

## 4 · ComfyUI neu starten

Zum Beispiel nach dem Installieren neuer Custom Nodes über den Manager. Der Neustart dauert etwa 20–60 Sekunden.

In [ ]:
run(f"{BIN}/restart_comfy.sh")
import time; time.sleep(20)
run(f"tail -n 20 {WS}/comfyui.log")

## 5 · Basis-Modelle erneut prüfen und laden

Zum Beispiel nach einem abgebrochenen Download oder wenn du eine andere Variante haben willst. Vorhandene Dateien werden übersprungen.

Varianten: `v4-turbo-int8` (Standard), `v4-turbo-int4`, `v4-turbo-int4b`, `v4-int8`, `v4-int4`, `v4-int4b`, `v3-turbo-int8`, `v3-turbo-int4`. Text-Encoder: `nvfp4`, `int4`, `int8`.

**Hinweis:** Bei einer anderen Variante passt diese Zelle den Workflow nicht an. Dazu den Pod neu starten, nachdem die Variable im Template geändert wurde, oder das Modell im Workflow selbst auswählen.

In [ ]:
VARIANTE = os.environ.get("DASIWA_VARIANT", "v4-turbo-int8")
TEXT_ENCODER = os.environ.get("TEXT_ENCODER", "nvfp4")
token = os.environ.get("CIVITAI_TOKEN") or os.environ.get("civitai_token") or getpass.getpass("CivitAI API-Key: ")
run(f"{BIN}/download_models.sh", env={"DASIWA_VARIANT": VARIANTE, "TEXT_ENCODER": TEXT_ENCODER, "CIVITAI_TOKEN": token, "CIVITAI_LORAS": ""})